[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Maxencegu/python-datascience-m1/blob/main/cours/cm07_git_github_et_outils/cm07_cours.ipynb)

# **CM07 — Git, GitHub & Outils professionnels**

**Python & Data Science · UPJV Amiens · L3 Économie**

---

Ce CM transforme votre façon de travailler : vous passerez d'un code "ça marche chez moi" à un code propre, sécurisé et reproductible.

> **Note** : les démonstrations sont réalisées **en direct**. Ce notebook est votre support de référence.

**Vous savez déjà** (TD01 / TD02 / CM01) : commandes Git de base, dépôts GitHub, projets avec `uv`.

**Ce CM ajoute les outils manquants :**
- **Branches Git** — expérimenter sans casser le code principal
- **GitHub CLI** — créer des dépôts depuis le terminal
- **Variables d'environnement** — garder les clés API hors du code
- **Fichiers `.env`** — la solution pratique avec `python-dotenv`
- **Ruff** — formater et analyser son code automatiquement en un seul outil

---
## 1. Pourquoi ces outils ?

Voici ce qui distingue un code "ça marche chez moi" d'un code professionnel :

| Problème courant | Solution |
|-----------------|----------|
| "J'ai écrasé mon code qui marchait" | **Branches Git** |
| "Ma clé API est sur GitHub, je suis hacké" | **Variables d'environnement + `.env`** |
| "Mon code est illisible / mal formaté" | **Ruff** |
| "Je ne sais pas recréer l'environnement" | **`uv.lock` + `uv sync`** (CM01) |
| "Je ne peux pas créer un repo depuis le terminal" | **GitHub CLI (`gh`)** |

Ces outils sont **indispensables en entreprise** et vous seront demandés dès votre premier stage.

---
## 2. Git professionnel

### 2.1 Le fichier `.gitignore`

`.gitignore` indique à Git quels fichiers **ne jamais suivre**. C'est la première chose à créer.

```
# .gitignore  (à la racine du projet)
.venv/          ← environnement virtuel (peut dépasser 1 Go)
.env            ← clés API et secrets
__pycache__/    ← fichiers compilés Python
*.pyc
.DS_Store       ← fichier macOS inutile
```

> **Règle critique** : créer `.gitignore` **avant** le premier `git add .`
> `uv init` le crée automatiquement — vérifiez qu'il contient `.env`.

---

### 2.2 Branches — expérimenter sans risque

Une branche est une **copie parallèle de votre code** où vous expérimentez sans toucher à la version stable.

```
main ──●──●──●──────────────●   ← version stable
            │               ↑
            └──●──●──●──────    ← branche dev_feature
```

**Commandes essentielles :**
```bash
git checkout -b dev_feature     # créer et basculer sur une branche
# ... travailler ...
git add . && git commit -m "..."
git push origin dev_feature     # pousser la branche
# → Ouvrir une Pull Request sur GitHub

git checkout main               # revenir sur main
git merge dev_feature           # fusionner le travail fini
```

> Vous utilisez déjà ce workflow en TD (`dev_tdXX` → Pull Request → main).

---

### 2.3 Workflow quotidien

```bash
git pull                        # récupérer les dernières modifications
# ... modifier les fichiers ...
git add .
git commit -m "Description claire du changement"
git push
```

---

### 2.4 Git dans VS Code

VS Code a Git intégré — tout se fait avec des clics.

| Opération | Terminal | VS Code |
|-----------|----------|---------|
| Voir les changements | `git status` | Panneau Source Control `Ctrl+Shift+G` |
| Stager | `git add .` | Cliquer `+` sur les fichiers |
| Committer | `git commit -m "..."` | Message + `Ctrl+Entrée` |
| Pousser / tirer | `git push` / `git pull` | Bouton sync ↑↓ (barre d'état) |

> Le bouton `↑2 ↓0` en bas de VS Code : ↑ = commits à pousser, ↓ = commits à récupérer.

In [ ]:
# Résumé des commandes Git essentielles (à exécuter dans Git Bash)

print("""
# === Nouveau projet ===
uv init mon-projet
cd mon-projet
git init
git add .
git commit -m "Initial commit"

# === Connexion à GitHub (GitHub CLI) ===
gh repo create mon-projet --private --source=. --remote=origin --push

# === Workflow quotidien ===
git pull                                          # récupérer les mises à jour
# ... modifier le code ...
git add .                                         # stager tous les changements
git commit -m "Ajouter analyse des ventes T3"    # commiter
git push                                          # envoyer sur GitHub

# === Branches ===
git checkout -b dev_feature                       # créer une branche
# ... expérimenter ...
git checkout main                                 # retour sur main
git merge dev_feature                             # fusionner
""")

---
## 3. GitHub CLI — créer des dépôts depuis le terminal

`gh` est l'outil officiel GitHub. Il permet de tout faire sans navigateur.

### 3.1 Installation

```bash
# Windows (Git Bash)
winget install --id GitHub.cli

# macOS
brew install gh
```

### 3.2 Connexion

```bash
gh auth login
# → GitHub.com → HTTPS → Authentifier via le navigateur
```

### 3.3 Commandes essentielles

```bash
# Créer un dépôt privé et pousser le projet actuel en une commande
gh repo create mon-projet --private --source=. --remote=origin --push

# Créer un dépôt public
gh repo create mon-projet --public --source=. --remote=origin --push

# Lister ses dépôts
gh repo list

# Ouvrir le dépôt dans le navigateur
gh repo view --web

# Créer une Pull Request
gh pr create --title "Ajouter analyse T3" --body "Description de la PR"

# Voir les Pull Requests ouvertes
gh pr list
```

> **Vous utilisez déjà `gh`** depuis la configuration du TD02.

---
## 4. Variables d'environnement & secrets

### 4.1 Le problème

```python
# ❌ NE JAMAIS FAIRE ÇA
import openai
client = openai.OpenAI(api_key="sk-abc123xyz789...")  # visible sur GitHub !
```

Si vous poussez ce code sur GitHub, votre clé API est exposée.
Des robots scannent GitHub en permanence et volent les clés en quelques **minutes**.

---

### 4.2 La solution — variables d'environnement

Une **variable d'environnement** est une valeur stockée dans votre système, pas dans votre code.

```python
# ✅ La bonne façon
import os
api_key = os.environ.get("OPENAI_API_KEY")  # lit depuis le système
```

La clé est dans votre **système local**, pas dans le code → elle n'ira jamais sur GitHub.

In [ ]:
import os

# Lire une variable d'environnement (None si elle n'existe pas)
api_key = os.environ.get("OPENAI_API_KEY")
print(f"Clé API : {api_key}")

# Avec une valeur par défaut
port = os.environ.get("PORT", "8000")
print(f"Port : {port}")

# Vérifier l'existence avant d'utiliser
if "DATABASE_URL" in os.environ:
    db_url = os.environ["DATABASE_URL"]
    print(f"Base de données : {db_url}")
else:
    print("Variable DATABASE_URL non définie")

# En Colab, on peut définir des variables manuellement pour tester
os.environ["NOM_ETUDIANT"] = "Alice"
print(f"Nom : {os.environ.get('NOM_ETUDIANT')}")

# Ne jamais utiliser os.environ["CLE"] sans vérifier — KeyError si absente
# Toujours préférer os.environ.get("CLE", "valeur_par_defaut")

---
### 4.3 Fichiers `.env` — la solution pratique

Taper `export OPENAI_API_KEY=sk-...` dans le terminal à chaque session est fastidieux.
La solution : un fichier `.env` chargé automatiquement au démarrage.

**Étape 1 — Installer python-dotenv**
```bash
uv add python-dotenv
```

**Étape 2 — Créer le fichier `.env` à la racine du projet**
```
# .env
OPENAI_API_KEY=sk-votre-vraie-clé-ici
DATABASE_URL=sqlite:///local.db
DEBUG=True
PORT=8000
```

**Règles d'écriture :**
- Une variable par ligne
- Pas d'espaces autour du `=`
- Noms en MAJUSCULES par convention
- `#` pour les commentaires

**Étape 3 — S'assurer que `.env` est dans `.gitignore`**
```
# .gitignore
.env          ← OBLIGATOIRE
.venv/
__pycache__/
```

> ⚠️ **`.env` ne doit jamais être commité.** Vérifiez votre `.gitignore` avant chaque `git push`.

In [ ]:
# Installation nécessaire : uv add python-dotenv
# En Colab, simuler le contenu d'un .env via os.environ pour la démonstration

import os
os.environ.setdefault("API_KEY", "demo-key-colab")
os.environ.setdefault("DEBUG", "True")
os.environ.setdefault("PORT", "8000")

from dotenv import load_dotenv

# load_dotenv() lit le fichier .env et charge ses variables dans os.environ
# En local : load_dotenv() suffit
# En Colab  : les variables simulées ci-dessus prennent le relais
load_dotenv()

# Récupérer les variables
api_key = os.environ.get("API_KEY")
debug   = os.environ.get("DEBUG", "False")
port    = os.environ.get("PORT", "8000")

print(f"API Key  : {api_key}")
print(f"Debug    : {debug}")
print(f"Port     : {port}")

# Pattern sécurisé : vérifier les variables critiques au démarrage
def verifier_config(variables_requises: list[str]) -> None:
    manquantes = [v for v in variables_requises if not os.environ.get(v)]
    if manquantes:
        raise EnvironmentError(f"Variables manquantes dans .env : {manquantes}")
    print("✅ Configuration complète")

verifier_config(["API_KEY", "PORT"])

---
### 4.4 `.env.example` — documenter sans exposer

Créez un fichier `.env.example` que vous **commitez** sur GitHub.
Il montre aux collaborateurs quelles variables définir, sans exposer vos vraies valeurs.

```
# .env.example  ← ce fichier est sur GitHub
OPENAI_API_KEY=your-api-key-here
DATABASE_URL=sqlite:///local.db
DEBUG=True
PORT=8000
```

**Workflow quand on clone un projet :**
```bash
git clone https://github.com/utilisateur/projet.git
cd projet
cp .env.example .env       # copier le modèle
# ... remplir .env avec ses vraies valeurs ...
uv sync                    # installer les dépendances
uv run main.py             # démarrer
```

**Résumé des fichiers :**

| Fichier | Sur GitHub ? | Contenu |
|---------|-------------|---------|
| `.env` | ❌ Non (gitignore) | Vraies valeurs secrètes |
| `.env.example` | ✅ Oui | Modèle avec valeurs factices |

---
## 5. Qualité du code — Ruff

### 5.1 Linter et formatter : à quoi ça sert ?

| Outil | Rôle |
|-------|------|
| **Formatter** | Reformate automatiquement le code pour le rendre lisible et cohérent |
| **Linter** | Détecte les erreurs de style, les imports inutilisés, les bugs potentiels |

**Ruff** remplace en un seul outil ultra-rapide :
- `black` ← formatter (mentionné en CM01)
- `isort` ← tri des imports (mentionné en CM01)
- `pylint` / `flake8` ← linter

> Si vous avez installé `black` et `isort` en CM01, Ruff les remplace : `uv add --dev ruff`.

---

### 5.2 Installer l'extension VS Code

1. `Ctrl+Shift+X` → chercher **"Ruff"**
2. Installer l'extension officielle par **Astral Software**

---

### 5.3 Activer "Format on save"

`Ctrl+,` → chercher **"format on save"** → cocher la case

Chaque `Ctrl+S` reformate désormais votre code automatiquement.

---

### 5.4 En ligne de commande

```bash
# Ajouter Ruff au projet
uv add --dev ruff

# Formater un fichier
uv run ruff format mon_fichier.py

# Analyser le code (linting)
uv run ruff check mon_fichier.py

# Corriger automatiquement les problèmes détectables
uv run ruff check --fix mon_fichier.py
```

In [ ]:
# Code mal formaté — typique quand on code vite
# Copiez ce code dans un fichier .py dans VS Code et sauvegardez avec Ctrl+S
# Ruff va le reformater automatiquement

import os
import sys
import json
def   calculer_total(articles):
    total=0
    for article in articles:
        total+=article['prix']*article['quantite']
    return total

panier=[{'nom':'café','prix':3.5,'quantite':2},{'nom':'croissant','prix':1.2,'quantite':3},{'nom':'sandwich','prix':5.9,'quantite':1}]
print(  calculer_total(panier)  )

print("Code non formaté — observer dans VS Code avec l'extension Ruff")

In [ ]:
# Code après formatage automatique par Ruff (Ctrl+S avec l'extension installée)
# Ruff a : trié les imports, corrigé l'espacement, reformaté les listes,
#           standardisé les guillemets, ajouté les lignes vides entre fonctions

import json
import os
import sys


def calculer_total(articles):
    total = 0
    for article in articles:
        total += article["prix"] * article["quantite"]
    return total


panier = [
    {"nom": "café", "prix": 3.5, "quantite": 2},
    {"nom": "croissant", "prix": 1.2, "quantite": 3},
    {"nom": "sandwich", "prix": 5.9, "quantite": 1},
]
print(calculer_total(panier))

In [ ]:
# Ruff détecte aussi les problèmes de code (linting)
# Ces avertissements apparaissent soulignés dans l'éditeur VS Code

import os    # F401 : import inutilisé si os n'est jamais utilisé

x = None
if x == None:  # E711 : utiliser "is None" plutôt que "== None"
    pass

resultat = 42  # F841 : variable assignée mais jamais lue

# Survoler le texte souligné dans VS Code → explication + suggestion de correction
print("Voir les soulignements Ruff dans VS Code pour chaque problème ci-dessus")

---
## 6. Workflow complet — nouveau projet de A à Z

### Étape 1 — Créer le projet
```bash
uv init mon-analyse
cd mon-analyse
```

### Étape 2 — Ajouter les dépendances
```bash
uv add pandas matplotlib requests python-dotenv
uv add --dev ruff ipykernel
```

### Étape 3 — Créer le fichier `.env`
```
# .env  (ne jamais commiter)
API_KEY=votre-clé-ici
DATABASE_URL=sqlite:///local.db
DEBUG=True
```

### Étape 4 — Créer `.env.example`
```
# .env.example  (commiter sur GitHub)
API_KEY=your-api-key-here
DATABASE_URL=sqlite:///local.db
DEBUG=True
```

### Étape 5 — Vérifier `.gitignore`
```
.env          ← présent ?
.venv/        ← présent ?
__pycache__/  ← présent ?
```

### Étape 6 — Premier commit et push
```bash
git add .
git commit -m "Initial commit"
gh repo create mon-analyse --private --source=. --remote=origin --push
```

### Étape 7 — Workflow quotidien
```bash
git pull                                          # récupérer les mises à jour
# ... coder (Ruff formate à chaque Ctrl+S) ...
uv add nouveau-paquet                             # si besoin
git add .
git commit -m "Ajouter analyse trimestrielle T3"
git push
```

In [ ]:
# Récapitulatif visuel du workflow

print("""
╔══════════════════════════════════════════════════════════════╗
║               WORKFLOW NOUVEAU PROJET                        ║
╠══════════════════════════════════════════════════════════════╣
║  uv init mon-projet                    Créer le projet       ║
║  cd mon-projet && code .              Ouvrir VS Code         ║
║  uv add pandas requests python-dotenv  Dépendances           ║
║  uv add --dev ruff ipykernel           Outils de dev         ║
║  Créer .env avec les vraies clés       Secrets locaux        ║
║  Créer .env.example avec fausses       À commiter            ║
║  git add . && git commit -m "Initial"  Premier commit        ║
║  gh repo create ... --push             Sur GitHub            ║
╠══════════════════════════════════════════════════════════════╣
║               WORKFLOW QUOTIDIEN                             ║
╠══════════════════════════════════════════════════════════════╣
║  git pull                              Récupérer les màj     ║
║  ... coder (Ruff formate à chaque save)                      ║
║  git add .                             Stager                ║
║  git commit -m "Message clair"         Commiter              ║
║  git push                              Pousser               ║
╚══════════════════════════════════════════════════════════════╝
""")

---
## 7. Récapitulatif

### Ce que vous savez maintenant

| Outil / Concept | Commande clé | Rôle |
|----------------|-------------|------|
| **Branches Git** | `git checkout -b nom` | Expérimenter sans risque |
| **GitHub CLI** | `gh repo create --push` | Créer un repo depuis le terminal |
| **`.gitignore`** | `.env` + `.venv/` | Protéger secrets et fichiers lourds |
| **`os.environ`** | `os.environ.get("CLE")` | Lire une variable d'environnement |
| **python-dotenv** | `load_dotenv()` | Charger les variables depuis `.env` |
| **`.env`** | (fichier local, gitignore) | Stocker les secrets localement |
| **`.env.example`** | (commité sur GitHub) | Documenter les variables nécessaires |
| **Ruff** | `uv run ruff format` | Formater et analyser le code |
| **Format on save** | (paramètre VS Code) | Formatage automatique à chaque `Ctrl+S` |

### Les règles d'or

1. **Jamais de secret dans le code** → toujours dans `.env`
2. **`.env` jamais sur GitHub** → toujours dans `.gitignore`
3. **`.env.example` toujours sur GitHub** → pour les collaborateurs
4. **Commiter souvent** → petits commits avec des messages descriptifs
5. **Branches pour expérimenter** → jamais directement sur `main`

### Prochain CM

**CM08 — Projet complet & Récapitulatif** : on met tout ensemble pour construire un projet de data science de bout en bout.